# Advanced: Multi-Strategy Identity Field Analysis with PAMOLA.CORE

**Goal**: Master all 3 identity analysis strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Exact matching for strict consistency checking
- **Strategy 2**: Fuzzy matching for typo-tolerant analysis
- **Strategy 3**: Hierarchical analysis for multi-level identifiers
- Calculate cross-matching and privacy risk metrics
- Identify optimal strategy for data quality assessment
- Production deployment patterns for identifier profiling

**Prerequisites:**
- Completed the simple notebook
- Understanding of identity consistency concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 02_identity_analyzer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.identity import IdentityAnalysisOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Load larger dataset from examples/data_examples/identity_analysis_data.csv
- Or generate realistic 1000-record sample with identifier issues
- Inspect data inconsistencies and cross-matching scenarios

**What you'll see:**
- File location and load status
- Dataset summary (1000 records, 7 columns)
- First 10 sample records showing identifier variations
- Column statistics (unique counts, ranges)
- Identifier distribution (~300 unique persons with 1-5 UIDs each)

**Dataset features:**
- 1000 records with multiple identifier scenarios
- Name variations (typos, abbreviations, formatting)
- UIDs with inconsistent assignments
- Cross-matching patterns for privacy analysis

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'identity_analysis_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate person base data
    first_names = ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 
                   'Henry', 'Iris', 'Jack', 'Karen', 'Leo', 'Mia', 'Nathan', 'Olivia',
                   'Peter', 'Quinn', 'Rachel', 'Steve', 'Tina', 'Uma', 'Victor', 'Wendy']
    last_names = ['Smith', 'Jones', 'Brown', 'Wilson', 'Davis', 'Miller', 'Taylor', 
                  'Anderson', 'Thomas', 'Jackson', 'White', 'Harris', 'Martin', 'Garcia',
                  'Martinez', 'Robinson', 'Clark', 'Rodriguez', 'Lewis', 'Lee', 'Walker']
    
    # Generate ~300 unique persons
    n_persons = 300
    persons = []
    for i in range(n_persons):
        persons.append({
            'person_id': i + 1,
            'first_name': np.random.choice(first_names),
            'last_name': np.random.choice(last_names),
            'birth_date': f"{np.random.randint(1970, 2000)}-{np.random.randint(1, 13):02d}-{np.random.randint(1, 29):02d}",
            'gender': np.random.choice(['M', 'F'])
        })
    
    # Generate records with varying UIDs per person
    records = []
    uid_counter = 1
    
    for person in persons:
        # Each person has 1-5 records
        n_records_per_person = np.random.choice([1, 2, 3, 4, 5], p=[0.2, 0.3, 0.25, 0.15, 0.1])
        
        # Generate UIDs for this person (sometimes same, sometimes different)
        uid_consistency = np.random.random()
        
        for j in range(n_records_per_person):
            # Decide UID assignment
            if uid_consistency > 0.7:  # 30% chance of inconsistent UID
                uid = f"UID{uid_counter:04d}"
                uid_counter += 1
            else:  # 70% chance of consistent UID
                if j == 0 or np.random.random() < 0.8:
                    uid = f"UID{uid_counter:04d}"
                    if j == 0:
                        uid_counter += 1
            
            # Add name variations (typos, abbreviations)
            first_name = person['first_name']
            if np.random.random() < 0.1:  # 10% typos
                if len(first_name) > 3:
                    first_name = first_name[:3] + first_name[4:]  # Remove one char
            elif np.random.random() < 0.15:  # 15% abbreviations
                first_name = first_name[0] + '.'
            
            last_name = person['last_name']
            if np.random.random() < 0.05:  # 5% typos
                if len(last_name) > 4:
                    last_name = last_name[:2] + last_name[3:]  # Remove one char
            
            records.append({
                'record_id': len(records) + 1,
                'UID': uid,
                'first_name': first_name,
                'last_name': last_name,
                'birth_date': person['birth_date'],
                'gender': person['gender'],
                'true_person_id': person['person_id']  # Ground truth for validation
            })
    
    # Trim to exactly 1000 records
    if len(records) > n_records:
        records = records[:n_records]
    
    df = pd.DataFrame(records)
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records (showing identifier variations):")
print(df.head(10))

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n🆔 Identifier Structure:")
print(f"  Unique UIDs: {df['UID'].nunique()}")
print(f"  Avg records per UID: {len(df) / df['UID'].nunique():.2f}")
if 'true_person_id' in df.columns:
    print(f"  True persons: {df['true_person_id'].nunique()}")
    print(f"  Avg UIDs per person: {df.groupby('true_person_id')['UID'].nunique().mean():.2f}")

print(f"\n💡 Perfect dataset for testing all 3 identity analysis strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field assignments for analysis
   - `uid_field`: Primary identifier (default: "UID")
   - `reference_fields`: Identity-defining fields (default: ["first_name", "last_name", "birth_date"])
   - `entity_field`: Optional entity ID (default: "record_id")
2. Run to validate fields and setup environment

**What you'll see:**
- UID field validation (✓ exists, unique values, sample)
- Reference fields validation (✓ all fields with unique counts)
- Entity field validation (✓ or skip notice)
- Task directory created (✓)
- TaskReporter initialized (✓)
- Config kwargs ready (✓)
- DataSource created (✓)

**This setup will be reused for all 3 strategies**

**Field configuration pattern:**
```python
FIELD_CONFIG = {
    "uid_field": "UID",              # Identifier to analyze
    "reference_fields": [            # Identity definition
        "first_name",
        "last_name",
        "birth_date"
    ],
    "entity_field": "record_id"      # Optional entity ID
}
```

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "uid_field": "UID",              # Primary identifier to analyze
    "reference_fields": [            # Fields defining identity
        "first_name",
        "last_name",
        "birth_date"
    ],
    "entity_field": "record_id"      # Optional entity-level ID
}

# Validate UID field exists
print("📋 Validating field configuration...\n")
uid_field = FIELD_CONFIG["uid_field"]
if uid_field not in df.columns:
    raise ValueError(
        f"❌ UID field '{uid_field}' not found!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update FIELD_CONFIG['uid_field']"
    )

print(f"✓ UID field: '{uid_field}'")
print(f"  Unique values: {df[uid_field].nunique()}")
print(f"  Sample values: {list(df[uid_field].unique()[:5])}")

# Validate reference fields
reference_fields = FIELD_CONFIG["reference_fields"]
valid_refs = [f for f in reference_fields if f in df.columns]
missing_refs = [f for f in reference_fields if f not in df.columns]

if not valid_refs:
    raise ValueError(
        f"❌ No reference fields found!\n"
        f"Looking for: {', '.join(reference_fields)}\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update FIELD_CONFIG['reference_fields']"
    )

if missing_refs:
    print(f"\n⚠️  Warning: Some reference fields not found: {', '.join(missing_refs)}")

print(f"\n✓ Reference fields ({len(valid_refs)}):")
for field in valid_refs:
    print(f"  - {field:20s}: {df[field].nunique()} unique values")

# Validate entity field (optional)
entity_field = FIELD_CONFIG["entity_field"]
if entity_field and entity_field not in df.columns:
    print(f"\n⚠️  Entity field '{entity_field}' not found - distribution analysis will be skipped")
    entity_field = None
elif entity_field:
    print(f"\n✓ Entity field: '{entity_field}'")
    print(f"  Unique values: {df[entity_field].nunique()}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy_identity",
    description="Multi-strategy identity analysis",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Exact Matching

**How to use:**
- Uses exact field comparison for strict consistency
- Best for clean data with standardized formatting
- Run to execute exact matching strategy

**Key parameters:**
- `fuzzy_matching=False`: Exact field comparison
- `check_cross_matches=True`: Detect same person with multiple UIDs
- `top_n=15`: Number of examples to include in analysis

**What you'll see:**
- Configuration confirmation
- Progress bar: validate → stats → consistency → distribution → cross-match → visualize → save
- Completion message with execution time (2-5 seconds)
- Quick stats (unique identifiers, consistency %, cross-matches)

**Note:** Exact matching requires perfect field matches - typos counted as different identities

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: EXACT MATCHING")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Exact",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = IdentityAnalysisOperation(
    # Core parameters
    uid_field=FIELD_CONFIG["uid_field"],
    reference_fields=valid_refs,
    id_field=entity_field,
    
    # Strategy: Exact matching
    fuzzy_matching=False,
    min_similarity=0.8,  # Not used in exact mode
    
    # Analysis parameters
    top_n=15,
    check_cross_matches=True,
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  UID field:            {operation_s1.uid_field}")
print(f"  Reference fields:     {len(operation_s1.reference_fields)}")
print(f"  Fuzzy matching:       {operation_s1.fuzzy_matching}")
print(f"  Check cross-matches:  {operation_s1.check_cross_matches}")

print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_exact',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Extract key metrics
if result_s1.metrics:
    print(f"\n📊 Quick Stats:")
    print(f"   Unique identifiers:   {result_s1.metrics.get('unique_identifiers', 0):,}")
    print(f"   Match percentage:     {result_s1.metrics.get('match_percentage', 0):.2f}%")
    print(f"   Cross-matches:        {result_s1.metrics.get('total_cross_matches', 0):,}")

## STRATEGY 2: Fuzzy Matching

**How to use:**
- Uses fuzzy string matching for typo tolerance
- Best for real-world data with entry errors
- Run to execute fuzzy matching strategy

**Key parameters:**
- `fuzzy_matching=True`: Enable fuzzy string comparison
- `min_similarity=0.8`: 80% similarity threshold for matches
- `check_cross_matches=True`: Detect fuzzy cross-matches

**What you'll see:**
- Configuration confirmation with fuzzy settings
- Progress bar: validate → stats → fuzzy consistency → distribution → fuzzy cross-match → visualize → save
- Completion message with execution time (5-10 seconds, slower than exact)
- Improved consistency stats (tolerates typos)

**Note:** Fuzzy matching detects similar strings - handles typos and formatting variations

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: FUZZY MATCHING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Fuzzy",
    unit="steps"
)

operation_s2 = IdentityAnalysisOperation(
    # Core parameters
    uid_field=FIELD_CONFIG["uid_field"],
    reference_fields=valid_refs,
    id_field=entity_field,
    
    # Strategy: Fuzzy matching
    fuzzy_matching=True,
    min_similarity=0.8,  # 80% similarity threshold
    
    # Analysis parameters
    top_n=15,
    check_cross_matches=True,
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  UID field:            {operation_s2.uid_field}")
print(f"  Reference fields:     {len(operation_s2.reference_fields)}")
print(f"  Fuzzy matching:       {operation_s2.fuzzy_matching}")
print(f"  Min similarity:       {operation_s2.min_similarity}")

start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_fuzzy',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Extract key metrics
if result_s2.metrics:
    print(f"\n📊 Quick Stats:")
    print(f"   Unique identifiers:   {result_s2.metrics.get('unique_identifiers', 0):,}")
    print(f"   Match percentage:     {result_s2.metrics.get('match_percentage', 0):.2f}%")
    print(f"   Cross-matches:        {result_s2.metrics.get('total_cross_matches', 0):,}")

## STRATEGY 3: Hierarchical Analysis

**How to use:**
- Uses subset of reference fields for broader matching
- Best for multi-level identifier hierarchies
- Run to execute hierarchical strategy

**Key parameters:**
- `fuzzy_matching=False`: Exact matching on reduced fields
- `reference_fields`: Uses only ["last_name", "birth_date"] for looser matching
- `check_cross_matches=True`: Detect hierarchical relationships

**What you'll see:**
- Configuration confirmation with reduced fields
- Progress bar: validate → stats → consistency → distribution → cross-match → visualize → save
- Completion message with execution time (3-6 seconds)
- Higher consistency due to fewer matching criteria

**Note:** Hierarchical analysis reveals family/group relationships by relaxing matching criteria

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: HIERARCHICAL ANALYSIS")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Hierarchical",
    unit="steps"
)

# Use subset of reference fields for hierarchical matching
hierarchical_refs = ['last_name', 'birth_date'] if all(f in valid_refs for f in ['last_name', 'birth_date']) else valid_refs[:2]

operation_s3 = IdentityAnalysisOperation(
    # Core parameters
    uid_field=FIELD_CONFIG["uid_field"],
    reference_fields=hierarchical_refs,  # Reduced field set
    id_field=entity_field,
    
    # Strategy: Hierarchical (exact on fewer fields)
    fuzzy_matching=False,
    min_similarity=0.8,
    
    # Analysis parameters
    top_n=15,
    check_cross_matches=True,
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  UID field:            {operation_s3.uid_field}")
print(f"  Reference fields:     {hierarchical_refs} (reduced set)")
print(f"  Fuzzy matching:       {operation_s3.fuzzy_matching}")
print(f"  Check cross-matches:  {operation_s3.check_cross_matches}")

start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_hierarchical',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Extract key metrics
if result_s3.metrics:
    print(f"\n📊 Quick Stats:")
    print(f"   Unique identifiers:   {result_s3.metrics.get('unique_identifiers', 0):,}")
    print(f"   Match percentage:     {result_s3.metrics.get('match_percentage', 0):.2f}%")
    print(f"   Cross-matches:        {result_s3.metrics.get('total_cross_matches', 0):,}")

## Step 4: Strategy Comparison

**How to use:**
- Review execution times and consistency metrics
- Compare cross-matching detection across strategies
- Identify optimal strategy for your data quality

**What you'll see:**
- Execution time per strategy (seconds)
- Consistency comparison (match percentages)
- Cross-match detection (counts per strategy)
- Total processing time

**Strategy selection guide:**
- **Exact**: Clean data + strict consistency requirements
- **Fuzzy**: Real-world data + typo tolerance needed
- **Hierarchical**: Family/group analysis + broader matching

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Exact):        {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (Fuzzy):        {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Hierarchical): {elapsed_s3:6.2f}s")
print(f"  Total:                     {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

# Compare consistency metrics
if result_s1.metrics and result_s2.metrics and result_s3.metrics:
    print(f"\n📈 Consistency Comparison:")
    match_s1 = result_s1.metrics.get('match_percentage', 0)
    match_s2 = result_s2.metrics.get('match_percentage', 0)
    match_s3 = result_s3.metrics.get('match_percentage', 0)
    
    print(f"  Strategy 1 (Exact):        {match_s1:6.2f}%")
    print(f"  Strategy 2 (Fuzzy):        {match_s2:6.2f}%")
    print(f"  Strategy 3 (Hierarchical): {match_s3:6.2f}%")
    
    print(f"\n🔍 Cross-Match Detection:")
    cross_s1 = result_s1.metrics.get('total_cross_matches', 0)
    cross_s2 = result_s2.metrics.get('total_cross_matches', 0)
    cross_s3 = result_s3.metrics.get('total_cross_matches', 0)
    
    print(f"  Strategy 1 (Exact):        {cross_s1:4d} cases")
    print(f"  Strategy 2 (Fuzzy):        {cross_s2:4d} cases")
    print(f"  Strategy 3 (Hierarchical): {cross_s3:4d} cases")

## Step 5: Detailed Artifact Review

**How to use:**
- Review all generated artifacts from each strategy
- Auto-loads NEWEST files from each category
- Displays metrics and visualizations inline

**What you'll see (per strategy):**
1. **Identifier Metrics**: Unique counts, coverage, relationships
2. **Consistency Metrics**: Match percentages, mismatch examples
3. **Distribution Metrics**: Records per identifier statistics
4. **Cross-Match Metrics**: Same person with different UIDs
5. **Visualizations**: Bar charts for all analyses (first 2 displayed)

**Note:** Only newest files shown. data_types metrics excluded automatically

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_exact', 'Strategy 1: Exact Matching'),
    ('strategy2_fuzzy', 'Strategy 2: Fuzzy Matching'),
    ('strategy3_hierarchical', 'Strategy 3: Hierarchical Analysis')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        # Get all metrics files
        all_metrics = list(metrics_dir.glob('*.json'))
        
        if all_metrics:
            # Filter out data_types
            filtered = [f for f in all_metrics if not f.name.startswith("data_types")]
            target_files = filtered if filtered else all_metrics
            
            # Sort by modification time
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            
            # Group by type (identifier, consistency, distribution, cross_match)
            metrics_by_type = {}
            for f in target_files:
                if 'identifier_metrics' in f.name:
                    metrics_by_type.setdefault('identifier', []).append(f)
                elif 'consistency_metrics' in f.name:
                    metrics_by_type.setdefault('consistency', []).append(f)
                elif 'distribution_metrics' in f.name:
                    metrics_by_type.setdefault('distribution', []).append(f)
                elif 'cross_match_metrics' in f.name:
                    metrics_by_type.setdefault('cross_match', []).append(f)
            
            # Display one from each type
            for metric_type, files in metrics_by_type.items():
                if files:
                    latest = files[0]
                    print(f"\n📄 {metric_type.upper()} METRICS: {latest.name}")
                    
                    try:
                        with open(latest, 'r') as f:
                            data = json.load(f)
                            metrics = data.get('metrics', data)
                        
                        # Display key metrics based on type
                        if metric_type == 'identifier':
                            print(f"   Unique identifiers:   {metrics.get('unique_identifiers', 0):,}")
                            print(f"   Coverage:             {metrics.get('coverage_percentage', 0):.2f}%")
                            print(f"   Uniqueness ratio:     {metrics.get('uniqueness_ratio', 0):.4f}")
                        elif metric_type == 'consistency':
                            print(f"   Match percentage:     {metrics.get('match_percentage', 0):.2f}%")
                            print(f"   Inconsistent combos:  {metrics.get('inconsistent_combinations', 0):,}")
                        elif metric_type == 'distribution':
                            print(f"   Avg count per ID:     {metrics.get('avg_count', 0):.2f}")
                            print(f"   Max count per ID:     {metrics.get('max_count', 0)}")
                        elif metric_type == 'cross_match':
                            print(f"   Total cross-matches:  {metrics.get('total_cross_matches', 0):,}")
                    except Exception as e:
                        print(f"   ⚠️  Error: {e}")
    
    # 2. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Privacy Risk Assessment

**How to use:**
- Calculate privacy risk based on identifier patterns
- Compare risk across strategies
- Run AFTER all 3 strategies complete

**What you'll see:**
- Identifier uniqueness (per strategy)
- Cross-match risk assessment
- Privacy recommendations

**Privacy risk levels:**
- Uniqueness > 0.9: High re-identification risk (direct identifier)
- Uniqueness 0.5-0.9: Medium risk (quasi-identifier)
- Uniqueness < 0.5: Low risk (grouping attribute)
- Cross-matches > 20%: Data quality issues or privacy concerns

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY RISK ASSESSMENT")
print("=" * 80 + "\n")

# Check if all strategies completed
try:
    if not (result_s1.metrics and result_s2.metrics and result_s3.metrics):
        print("⚠️  Run all 3 strategies first!")
    else:
        strategies = [
            ('Strategy 1 (Exact)', result_s1.metrics),
            ('Strategy 2 (Fuzzy)', result_s2.metrics),
            ('Strategy 3 (Hierarchical)', result_s3.metrics)
        ]
        
        for name, metrics in strategies:
            print(f"📊 {name}:")
            
            # Get uniqueness ratio
            uniqueness = metrics.get('uniqueness_ratio', 0)
            total_records = metrics.get('total_records', 0)
            unique_ids = metrics.get('unique_identifiers', 0)
            cross_matches = metrics.get('total_cross_matches', 0)
            cross_match_rate = (cross_matches / total_records * 100) if total_records > 0 else 0
            
            print(f"\n   Identifier Characteristics:")
            print(f"      Uniqueness ratio:      {uniqueness:.4f}")
            print(f"      Total records:         {total_records:,}")
            print(f"      Unique identifiers:    {unique_ids:,}")
            print(f"      Cross-matches:         {cross_matches:,} ({cross_match_rate:.1f}%)")
            
            # Risk assessment
            print(f"\n   Risk Assessment:")
            if uniqueness > 0.9:
                print(f"      ⚠️  HIGH re-identification risk")
                print(f"      → Acts as direct identifier")
                print(f"      → Requires strong protection")
            elif uniqueness > 0.5:
                print(f"      ⚠️  MEDIUM re-identification risk")
                print(f"      → Acts as quasi-identifier")
                print(f"      → Combine with other attributes for privacy risk")
            else:
                print(f"      ✓ LOW re-identification risk")
                print(f"      → Acts as grouping attribute")
                print(f"      → Safe for aggregation")
            
            if cross_match_rate > 20:
                print(f"\n      ⚠️  HIGH cross-match rate detected")
                print(f"      → Data quality issues OR")
                print(f"      → Multiple identities per person")
            
            print("\n" + "-" * 80 + "\n")
        
        print(f"\n💡 Choose strategy based on your privacy requirements and data quality")
        
except NameError:
    print("⚠️  Run Step 3 (Setup Environment) and all strategies first!")

## Step 7: Export Analysis Results

**How to use:**
- Export summary metrics from all strategies
- Each strategy gets its own summary file
- Run AFTER all 3 strategies complete

**What you'll see (per strategy):**
- Summary statistics (identifiers, consistency, cross-matches)
- Export confirmation with file path
- File size

**Output structure:**
```
advanced_output/
├── strategy1_exact/
│   ├── metrics/              # Full metrics JSONs
│   ├── visualizations/       # Charts
│   └── summary.json          # Key stats
├── strategy2_fuzzy/
│   └── ...
└── strategy3_hierarchical/
    └── ...
```

In [ ]:
print("=" * 80)
print("📦 EXPORTING ANALYSIS RESULTS FROM ALL STRATEGIES")
print("=" * 80)

# Check if all strategies completed
try:
    if not (result_s1.metrics and result_s2.metrics and result_s3.metrics):
        print("\n❌ Run all 3 strategies first!")
    else:
        export_base_dir = task_dir
        print(f"\n📂 Export directory: {export_base_dir}\n")
        
        strategies = [
            ('strategy1_exact', 'Exact Matching', result_s1.metrics),
            ('strategy2_fuzzy', 'Fuzzy Matching', result_s2.metrics),
            ('strategy3_hierarchical', 'Hierarchical Analysis', result_s3.metrics)
        ]
        
        for dir_name, strategy_name, metrics in strategies:
            print("=" * 80)
            print(f"📊 STRATEGY: {strategy_name}")
            print("=" * 80)
            
            strategy_dir = export_base_dir / dir_name
            strategy_dir.mkdir(parents=True, exist_ok=True)
            
            # Create summary
            summary = {
                'strategy': strategy_name,
                'timestamp': datetime.now().isoformat(),
                'statistics': {
                    'total_records': metrics.get('total_records', 0),
                    'unique_identifiers': metrics.get('unique_identifiers', 0),
                    'uniqueness_ratio': metrics.get('uniqueness_ratio', 0),
                    'coverage_percentage': metrics.get('coverage_percentage', 0),
                    'match_percentage': metrics.get('match_percentage', 0),
                    'total_cross_matches': metrics.get('total_cross_matches', 0)
                }
            }
            
            # Save summary
            summary_path = strategy_dir / 'summary.json'
            try:
                with open(summary_path, 'w') as f:
                    json.dump(summary, f, indent=2)
                print(f"\n✅ Saved: {summary_path}")
                print(f"   Size: {summary_path.stat().st_size / 1024:.1f} KB")
            except PermissionError:
                print(f"\n⚠️  Cannot save (file may be open): {summary_path}")
            
            # Display summary
            print(f"\n📊 Summary:")
            print(f"   Total records:        {summary['statistics']['total_records']:,}")
            print(f"   Unique identifiers:   {summary['statistics']['unique_identifiers']:,}")
            print(f"   Uniqueness ratio:     {summary['statistics']['uniqueness_ratio']:.4f}")
            print(f"   Coverage:             {summary['statistics']['coverage_percentage']:.2f}%")
            print(f"   Consistency:          {summary['statistics']['match_percentage']:.2f}%")
            print(f"   Cross-matches:        {summary['statistics']['total_cross_matches']:,}")
            print()
        
        # Overall summary
        print("\n" + "=" * 80)
        print("✅ EXPORT COMPLETE")
        print("=" * 80)
        print(f"\n📂 All summaries saved to: {export_base_dir}")
        
        if 'elapsed_s1' in locals():
            total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
            print(f"⏱️  Total processing time: {total_time:.2f}s")
        
except NameError:
    print("❌ Run Step 3 (Setup Environment) and all strategies first!")

## 🎯 Summary

**Accomplished:**
- ✅ 3 strategies implemented and compared
- ✅ Identifier consistency metrics calculated
- ✅ Cross-matching patterns identified
- ✅ Privacy risk assessment completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Exact matching reveals true data quality issues
- Fuzzy matching improves tolerance for entry errors
- Hierarchical analysis reveals group relationships
- High cross-matches indicate data quality problems
- Uniqueness ratio measures re-identification risk

**Strategy recommendations:**
- **Use Exact** when: Clean data, standardized formatting, strict compliance
- **Use Fuzzy** when: Real-world data, typos expected, quality assessment
- **Use Hierarchical** when: Family/group analysis, multi-level identifiers, broader patterns

**Next steps:**
- Test with your own identifier fields
- Tune similarity thresholds for optimal detection
- Integrate with data quality pipeline
- Deploy to production monitoring

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)